# 00 · Orientation

**Read first:** the book's Preface. Then this, then `LEARNING_PATH.md`.

---

## What you'll be able to do after this

- Run the tests and know what a green suite is actually telling you.
- Find any formula from Chapters 1–12 in the package, and the chapter it came from.
- Know where the simplifications are recorded, so you never trust a number without knowing what it assumes.

## What this repository is

A companion to *FX Derivatives Trader School* (Giles Jewitt, Wiley 2015), covering **Chapters 1–12 and Practicals A–F**. The book's practicals are written for Excel and VBA; these are the same tasks, same steps, same test cases, in Python.

You need the book. This is a companion, not a replacement — the code and explanations here are original, and the book is cited by chapter and practical number throughout.

| Practical | Title | Chapter | Module |
|---|---|---|---|
| A | Trading Simulator | 3 | `fxds/simulator/` |
| B | Numerical Integration Pricer | 5 | `fxds/numerical.py` |
| C | Black-Scholes Pricer | 5–6 | `fxds/blackscholes.py` |
| D | Tenor Dates | 10 | `fxds/dates.py` |
| E | ATM Curve | 11 | `fxds/atm_curve.py` |
| F | Volatility Smile | 12 | `fxds/smile.py` |
| — | Assembled surface *(not in the book)* | D+E+F | `fxds/surface.py` |

Practical G onwards (Chapter 13+) is out of scope.

## The one structural rule

**The package implements. The notebooks demonstrate.**

Notebooks import from `fxds/` and never redefine the maths inline. So there is exactly one implementation of each formula, the tests cover the thing the notebooks actually run, and if you fix a bug you fix it once.

When you see a formula in a notebook it's in a markdown cell, being explained. When you see it evaluated, it's an import.

In [1]:
import fxds
from fxds.conventions import OptionType
from fxds.blackscholes import price, delta_closed_form, vega_market
from fxds.numerical import price_vanilla

print(f"fxds version {fxds.__version__}")
print()

# The book's reference contract, from Practical C, Task A, Step 2.
BOOK = dict(spot=1.0, strike=1.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
print("Practical C acceptance test — S = K = 1.0, T = 1.0, sigma = 10%, zero rates")
print(f"  price   {price(OptionType.CALL, **BOOK):.6f}   book: 0.0399")
print(f"  delta   {delta_closed_form(OptionType.CALL, **BOOK):.4%}     book: close to 50%")
print(f"  vega    {vega_market(**BOOK):.5%}    book: a shade under 0.40%")

fxds version 0.1.0

Practical C acceptance test — S = K = 1.0, T = 1.0, sigma = 10%, zero rates
  price   0.039878   book: 0.0399
  delta   51.9939%     book: close to 50%
  vega    0.39844%    book: a shade under 0.40%


## Units, decided once

Getting these wrong is the most common way to produce a number that looks plausible and is wrong by a factor of 100. They are fixed across the whole package:

| Quantity | Internal representation | Not |
|---|---|---|
| Volatility | decimal — `0.085` | `8.5` |
| Interest rates | continuously compounded decimal | annual/simple |
| Delta | decimal fraction of CCY1 notional | percent |
| Spot, strike, forward | CCY2 per CCY1 | anything else |
| Option price | CCY2 pips, unless the name says otherwise | CCY1% |

Percent signs go on at **display time only**. Market data providers convert at their boundary and nowhere else — there's a test that fails if any provider hands back a volatility above 1.0.

In [2]:
from fxds.conventions import CurrencyPair, ccy2_pips_to_ccy1_pct, ccy2_pips_to_ccy2_cash

pair = CurrencyPair.parse("EURUSD")
pips = price(OptionType.CALL, 1.30, 1.32, 0.5, 0.005, 0.025, 0.085)

print(f"{pair.name}  pip size {pair.pip}   premium paid in {pair.premium_side.value}"
      f" ({pair.premium_side.market_name})")
print()
print(f"  price in CCY2 pips   {pips:.6f}")
print(f"  as a pip count       {pips / pair.pip:.1f} pips")
print(f"  as CCY1%             {ccy2_pips_to_ccy1_pct(pips, 1.30):.4%}")
print(f"  USD cash on EUR10m   {ccy2_pips_to_ccy2_cash(pips, 10e6):,.2f}")

EUR/USD  pip size 0.0001   premium paid in CCY2 (RHS)

  price in CCY2 pips   0.027866
  as a pip count       278.7 pips
  as CCY1%             2.1436%
  USD cash on EUR10m   278,663.35


## Running the tests

Every test case the book states is an assertion, with a comment naming the practical it came from.

In [3]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q", "--no-header",
     "--color=no", "-p", "no:cacheprovider"],
    capture_output=True, text=True, cwd="..",
)
# Just the summary line - the dots are not the interesting part.
print("\n".join(line for line in result.stdout.splitlines() if "passed" in line or "failed" in line))

419 passed, 1 skipped in 4.59s


### The one test worth understanding

`tests/test_cross_validation.py` checks that **Practical B and Practical C agree**.

They're independent. The integration pricer builds a distribution and sums payoff × probability; it has never heard of Garman–Kohlhagen. The closed-form pricer evaluates one algebraic expression; it has never heard of a grid. They share only the inputs and the log-normal assumption.

Agreement across a wide range of parameters is strong evidence both are right — and it reframes the closed form as what it actually is: **the analytic solution to the integral the other one computes numerically.**

In [4]:
args = dict(spot=100.0, strike=100.0, T=1.0, r_ccy1=0.0, r_ccy2=0.0, sigma=0.10)
analytic = price(OptionType.CALL, **args)
numeric = price_vanilla(OptionType.CALL, **args, sd_step=0.01).value_ccy2_pips

print(f"closed form (Practical C)   {analytic:.8f}")
print(f"integration (Practical B)   {numeric:.8f}")
print(f"relative difference         {abs(numeric - analytic) / analytic:.2e}")

closed form (Practical C)   3.98776117
integration (Practical B)   3.98778378
relative difference         5.67e-06


## Where things are

```
fxds/            one module per practical, plus shared conventions and plotting
notebooks/       one per stage of the learning path
notes/           source_notes.md, glossary.md, deviations.md
tests/           one per module, plus the cross-validation test
docs/            ticker verification procedure
```

Three files to know about before you read any code:

- **`notes/source_notes.md`** — the working spec. Every formula, convention and test value from Chapters 1–12, chapter by chapter, ending in a table of all 22 acceptance tests.
- **`notes/deviations.md`** — every place this implementation departs from the book, and why. Includes two transcription defects in the book's own printed VBA.
- **`notes/glossary.md`** — every term of art in plain English, with the chapter it first appears in.

## On register, and on trusting numbers

The book keeps its mathematics at a deliberate "advanced high school" level and is candid that it simplifies — particularly around credit risk and how interest rate markets actually work. Those caveats are carried through here rather than quietly dropped.

Where the book leaves a convention unspecified and I wasn't certain, the common convention is implemented, flagged in the docstring, and listed in `notes/deviations.md`. **Nothing in this repository is invented and presented as market fact.** If a docstring says a convention is assumed, treat it as assumed.

The biggest simplification to know about up front: the volatility smile here is the **outright-delta Malz model**, not the broker fly the interbank market actually trades. Chapter 12 spends several pages on why those differ. Practical F doesn't implement it and neither does this repo — but `fxds/smile.py`, `fxds/surface.py` and notebook 11 all say so.

In [5]:
from pathlib import Path
notes = Path("../notes/deviations.md").read_text()
print(f"notes/deviations.md — {len(notes.splitlines())} lines\n")
print("\n".join(line for line in notes.splitlines() if line.startswith("## ")))

notes/deviations.md — 117 lines

## Practical A — Trading Simulator
## Practical B — Numerical Integration Pricer
## Practical C — Black-Scholes Pricer
## Practical D — Tenor Dates
## Practical E — ATM Curve
## Practical F — Volatility Smile
## Assembled volatility surface (`surface.py`)
## Repository-wide


## Where next

Follow **`LEARNING_PATH.md`** — it gives the ordered route through book chapters and notebooks with honest time estimates (roughly 35–45 hours end to end; it is a fortnight of evenings, not a weekend).

The short version:

1. Chapters 1–2 → notebook 01
2. Chapter 3 + Practical A → notebook 02, then the Streamlit app
3. Chapter 4 → notebook 03
4. Chapter 5 + Practical B → **notebook 04**
5. Practical C + Chapter 6 → **notebooks 05 and 06**
6. Chapters 7–9 → notebooks 07 and 08
7. Chapter 10 + Practical D → notebook 09
8. Chapter 11 + Practical E → notebook 10
9. Chapter 12 + Practical F → notebook 11, then the assembled surface

The book's own advice in the Preface is worth repeating: do the practicals, do them all, do them in order, and don't reach for the finished spreadsheets unless you are completely stuck. The same applies here — reading `fxds/blackscholes.py` is not the same as having built it.